In [1]:
!pip install pandas numpy scikit-learn streamlit

In [2]:
import pandas as pd

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
data = pd.merge(ratings, movies, on='movieId')
data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [4]:
movie_matrix = data.pivot_table(index='title', columns='userId', values='rating')
movie_matrix.fillna(0, inplace=True)

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(movie_matrix)

In [6]:
import numpy as np

def recommend(movie_name):
    if movie_name not in movie_matrix.index:
        return ["Movie not found"]

    idx = movie_matrix.index.get_loc(movie_name)
    scores = list(enumerate(similarity[idx]))

    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]

    recommendations = [movie_matrix.index[i[0]] for i in sorted_scores]
    return recommendations

In [7]:
print(recommend("Toy Story (1995)"))

['Toy Story 2 (1999)', 'Jurassic Park (1993)', 'Independence Day (a.k.a. ID4) (1996)', 'Star Wars: Episode IV - A New Hope (1977)', 'Forrest Gump (1994)']


In [8]:
print(recommend("Jumanji (1995)	"))

['Movie not found']


In [10]:
print(recommend("Grumpier Old Men (1995)"))	

['Grumpy Old Men (1993)', 'Striptease (1996)', 'Nutty Professor, The (1996)', 'Twister (1996)', 'Father of the Bride Part II (1995)']


In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use genres instead of ratings
movies['genres'] = movies['genres'].str.replace('|', ' ')

cv = CountVectorizer()
genre_matrix = cv.fit_transform(movies['genres'])

genre_similarity = cosine_similarity(genre_matrix)

In [12]:
def recommend_content(movie_name):
    if movie_name not in movies['title'].values:
        return ["Movie not found"]

    idx = movies[movies['title'] == movie_name].index[0]
    scores = list(enumerate(genre_similarity[idx]))

    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:6]

    return [movies.iloc[i[0]]['title'] for i in sorted_scores]

In [13]:
print(recommend_content("Toy Story (1995)"))

['Antz (1998)', 'Toy Story 2 (1999)', 'Adventures of Rocky and Bullwinkle, The (2000)', "Emperor's New Groove, The (2000)", 'Monsters, Inc. (2001)']


In [14]:
# --- Collaborative (you already have) ---
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# movie_matrix: pivot table you created earlier
collab_sim = cosine_similarity(movie_matrix)

# --- Content (genres) ---
from sklearn.feature_extraction.text import CountVectorizer

movies['genres'] = movies['genres'].fillna('').str.replace('|', ' ', regex=False)

cv = CountVectorizer()
genre_matrix = cv.fit_transform(movies['genres'])
content_sim = cosine_similarity(genre_matrix)

# --- helper: map title -> index for both spaces ---
title_to_idx_collab = {title:i for i, title in enumerate(movie_matrix.index)}
title_to_idx_content = {title:i for i, title in enumerate(movies['title'])}

In [15]:
def recommend_hybrid(movie_name, top_n=5, alpha=0.6):
    """
    alpha = weight for collaborative (0–1)
    (1-alpha) = weight for content
    """
    if movie_name not in title_to_idx_content:
        return ["Movie not found"]

    # indices
    idx_c = title_to_idx_content[movie_name]

    # content scores
    content_scores = list(enumerate(content_sim[idx_c]))

    # collaborative scores (if movie exists in matrix)
    if movie_name in title_to_idx_collab:
        idx_cf = title_to_idx_collab[movie_name]
        collab_scores = list(enumerate(collab_sim[idx_cf]))
    else:
        collab_scores = [(i, 0) for i in range(len(content_scores))]

    # combine
    combined = []
    for i in range(len(content_scores)):
        s_content = content_scores[i][1]
        s_collab = collab_scores[i][1] if i < len(collab_scores) else 0
        score = alpha * s_collab + (1 - alpha) * s_content
        combined.append((i, score))

    # sort & skip itself
    combined = sorted(combined, key=lambda x: x[1], reverse=True)
    combined = [x for x in combined if movies.iloc[x[0]]['title'] != movie_name]

    # top N titles
    recs = [movies.iloc[i]['title'] for i, _ in combined[:top_n]]
    return recs

In [16]:
print(recommend_hybrid("Toy Story (1995)", top_n=5, alpha=0.6))

['Barely Lethal (2015)', 'Ponyo (Gake no ue no Ponyo) (2008)', 'Tale of Despereaux, The (2008)', 'Hotel Transylvania (2012)', "Emperor's New Groove, The (2000)"]


In [17]:
import streamlit as st
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

# load
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

# prep
data = pd.merge(ratings, movies, on='movieId')
movie_matrix = data.pivot_table(index='title', columns='userId', values='rating').fillna(0)

# similarities
collab_sim = cosine_similarity(movie_matrix)

movies['genres'] = movies['genres'].fillna('').str.replace('|', ' ', regex=False)
cv = CountVectorizer()
genre_matrix = cv.fit_transform(movies['genres'])
content_sim = cosine_similarity(genre_matrix)

title_to_idx_collab = {t:i for i, t in enumerate(movie_matrix.index)}
title_to_idx_content = {t:i for i, t in enumerate(movies['title'])}

def recommend_hybrid(movie_name, top_n=5, alpha=0.6):
    if movie_name not in title_to_idx_content:
        return ["Movie not found"]

    idx_c = title_to_idx_content[movie_name]
    content_scores = list(enumerate(content_sim[idx_c]))

    if movie_name in title_to_idx_collab:
        idx_cf = title_to_idx_collab[movie_name]
        collab_scores = list(enumerate(collab_sim[idx_cf]))
    else:
        collab_scores = [(i, 0) for i in range(len(content_scores))]

    combined = []
    for i in range(len(content_scores)):
        s_content = content_scores[i][1]
        s_collab = collab_scores[i][1] if i < len(collab_scores) else 0
        combined.append((i, 0.6 * s_collab + 0.4 * s_content))

    combined = sorted(combined, key=lambda x: x[1], reverse=True)
    combined = [x for x in combined if movies.iloc[x[0]]['title'] != movie_name]
    return [movies.iloc[i]['title'] for i, _ in combined[:top_n]]

# UI
st.title("🎬 Movie Recommender (Hybrid)")

movie = st.selectbox("Pick a movie", movies['title'].values)
alpha = st.slider("Preference (Collaborative vs Content)", 0.0, 1.0, 0.6)

if st.button("Recommend"):
    recs = recommend_hybrid(movie, top_n=5, alpha=alpha)
    st.subheader("Top 5 Recommendations:")
    for r in recs:
        st.write("•", r)

2026-05-07 19:21:28.329 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-07 19:21:28.408 
  command:

    streamlit run C:\Users\pruth\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-07 19:21:28.408 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-07 19:21:28.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-07 19:21:28.410 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-07 19:21:28.410 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-07 19:21:28.413 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-07 19:21:28.415 Thread 'MainThread': mi

In [18]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)